# Week 7 — Supervised Classification

This notebook covers the third project milestone:
- Load the 194-feature matrix extracted in notebook 02
- Split into train / test sets and scale features using **MiniLearn**
- Train four classifiers: KNN, Logistic Regression, Gaussian Naive Bayes, Decision Tree
- Evaluate each with accuracy, macro-averaged F1, and a full classification report
- Visualise per-class confusion matrices
- Produce a side-by-side summary table for comparison

In [ ]:
import sys
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# make minilearn importable from the notebooks directory
sys.path.insert(0, os.path.abspath('..'))

from minilearn.preprocessing import StandardScaler, train_test_split
from minilearn.classifiers import KNN, LogisticRegression, GaussianNaiveBayes, DecisionTreeClassifier
from minilearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 1. Load Feature Matrix

In [ ]:
df = pd.read_csv('../outputs/features.csv')

LABEL_COLS = ['filename', 'emotion', 'emotion_id', 'actor', 'gender', 'channel']
feature_cols = [c for c in df.columns if c not in LABEL_COLS]

X = df[feature_cols].values          # (2452, 194)  float64
y = df['emotion'].values             # (2452,)       string labels

EMOTION_ORDER = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

print(f'Feature matrix : {X.shape}')
print(f'Classes        : {np.unique(y).tolist()}')
print(f'Class counts   :')
for em in EMOTION_ORDER:
    print(f'  {em:10s}: {np.sum(y == em)}')

## 2. Train / Test Split and Feature Scaling

We hold out 20 % of samples as a test set.  The `StandardScaler` is **fit only on the
training set** and then applied to both splits — this prevents data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')
print(f'Feature mean after scaling (train): {X_train_s.mean():.4f}  (should be ~0)')
print(f'Feature std  after scaling (train): {X_train_s.std():.4f}  (should be ~1)')

## 3. Train and Evaluate Classifiers

Each classifier is wrapped in a helper that records training time, test accuracy, and
macro-F1.  We then inspect each one individually.

In [ ]:
classifiers = [
    ('KNN (k=5)',           KNN(n_neighbors=5)),
    ('Logistic Regression', LogisticRegression(lr=0.1, max_iter=500, random_state=RANDOM_STATE)),
    ('Gaussian Naive Bayes',GaussianNaiveBayes()),
    ('Decision Tree (d=10)',DecisionTreeClassifier(max_depth=10, random_state=None if True else 0)),
]

results = {}   # name -> {'clf', 'y_pred', 'acc', 'f1', 'train_time'}

for name, clf in classifiers:
    t0 = time.time()
    clf.fit(X_train_s, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = clf.predict(X_test_s)
    pred_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')

    results[name] = dict(clf=clf, y_pred=y_pred, acc=acc, f1=f1, train_time=train_time)
    print(f'{name:25s}  acc={acc:.3f}  macro-F1={f1:.3f}  '
          f'train={train_time:.1f}s  predict={pred_time:.1f}s')

## 4. Per-Classifier Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n{"=" * 55}')
    print(f'  {name}')
    print(f'{"=" * 55}')
    print(classification_report(
        y_test, res['y_pred'],
        labels=EMOTION_ORDER,
        target_names=EMOTION_ORDER
    ))

## 5. Confusion Matrices

Each cell (i, j) shows how many true-class-i samples were predicted as class j.
A perfect classifier has all mass on the diagonal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], labels=EMOTION_ORDER)
    # normalise rows to show recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=EMOTION_ORDER, yticklabels=EMOTION_ORDER,
        ax=ax, vmin=0, vmax=1, linewidths=0.4
    )
    ax.set_title(f'{name}\nacc={res["acc"]:.3f}  macro-F1={res["f1"]:.3f}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Normalised Confusion Matrices — RAVDESS 8-class SER', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/confusion_matrices.png')

## 6. Summary Comparison Table

In [ ]:
summary = pd.DataFrame([
    {
        'Classifier'  : name,
        'Test Accuracy': f"{res['acc']:.3f}",
        'Macro F1'    : f"{res['f1']:.3f}",
        'Train Time (s)': f"{res['train_time']:.1f}",
    }
    for name, res in results.items()
])

print(summary.to_string(index=False))
summary

## 7. Per-Class F1 Bar Chart

Showing how each classifier handles each emotion individually.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
palette = sns.color_palette('tab10', len(EMOTION_ORDER))

for ax, (name, res) in zip(axes, results.items()):
    per_class_f1 = f1_score(y_test, res['y_pred'], average=None)
    # f1_score(average=None) returns list aligned to sorted unique labels
    # reorder to EMOTION_ORDER
    unique_labels = sorted(np.unique(np.concatenate([y_test, res['y_pred']])))
    label_to_f1 = dict(zip(unique_labels, per_class_f1))
    f1_vals = [label_to_f1.get(em, 0.0) for em in EMOTION_ORDER]

    bars = ax.bar(EMOTION_ORDER, f1_vals, color=palette)
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_ylabel('F1 Score')
    ax.tick_params(axis='x', rotation=40)
    for bar, val in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Per-Class F1 Score by Classifier', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/per_class_f1.png')